This is just a test to install and test Ollama

In [5]:
import ollama

text = input()

response = ollama.chat(
    model="llama3.2",
    messages=[{"role": "user", "content": text}]
)

print(response["message"]["content"])

Not much! Just hanging out in the digital realm. It's great to meet you! Is there something I can help you with or would you like to chat?


Claude told me to try out OpenStreetMap(OSM) which has resturants, hotels, and landmarks for the entire world


In [23]:
import osmnx as ox

gdf = ox.features_from_place("New York City, New York, USA", tags={"tourism": "attraction"})
gdf.to_file("data/nyc_landmarks.geojson", driver="GeoJSON")

print(f"Restaurants: {len(gdf)}")

Restaurants: 266


I downloaded a NYC restaurant and landmark dataset, and now im testing it with Ollama

In [ ]:
import ollama, json
from pathlib import Path

folder_path = Path("data")

#Restaurants
with open(folder_path / "nyc_restaurants.geojson") as f:
    restaurant_data = json.load(f)

restaurant_names = [
    f["properties"].get("name", "Unknown")
    for f in restaurant_data["features"]
    if f["properties"].get("name")
]

#Landmarks
with open(folder_path / "nyc_landmarks.geojson") as f:
    landmark_data = json.load(f)

landmark_names = [
    f["properties"].get("name", "Unknown")
    for f in landmark_data["features"]
]
text = input()

prompt = f"""
You are an offline travel agent. Here are restaurants in NYC:
Restaurant: {restaurant_names}
Landmarks: {landmark_names}

Question: {text}
"""

response = ollama.chat(
    model = "llama3.2",
    messages = [{"role" : "user", "content" : prompt}]
)
print(response["message"]["content"])

You're looking for some unique landmarks near the pier! Based on the list, here are some interesting options:

1. **The Pier 55**: This iconic pier is a must-visit, especially at night when it's lit up. You can also explore the surrounding area, which features a variety of public art installations and beautiful gardens.
2. **The Sea Lion Store & Cafe**: This quirky landmark is home to a group of friendly sea lions that have made themselves comfortable on the pier. You can feed and interact with them, making for a fun and memorable experience.
3. **The Little Island**: This small island in the Hudson River is a peaceful oasis in the midst of the bustling city. Take a stroll around the island, explore the public art installations, and enjoy the beautiful views of the Manhattan skyline.
4. **The Bethesda Fountain**: Located in Central Park, just a short walk from the pier, this beautiful fountain is a stunning example of Beaux-Arts architecture. It's a great spot for people-watching and t

Now we are trying to use location to get the response

In [35]:
import geopandas as gpd
import ollama
from geopy.distance import geodesic

# Load data
restaurants = gpd.read_file("data/nyc_restaurants.geojson")

# Extract coordinates
restaurants["latitude"] = restaurants.geometry.to_crs(epsg=3857).centroid.to_crs(epsg=4326).y
restaurants["longitude"] = restaurants.geometry.to_crs(epsg=3857).centroid.to_crs(epsg=4326).x

restaurants = restaurants[["name", "cuisine", "addr:street", "latitude", "longitude"]].dropna(subset=["name"])

# NYC area coordinates
area_coordinates = {
    "midtown": (40.7549, -73.9840),
    "times square": (40.7580, -73.9855),
    "brooklyn": (40.6782, -73.9442),
    "queens": (40.7282, -73.7949),
    "manhattan": (40.7831, -73.9712),
    "bronx": (40.8448, -73.8648),
}

def filter_nearby(df, user_location, radius_km=1.0):
    df = df.copy()
    df["distance_km"] = df.apply(
        lambda row: geodesic(user_location, (row["latitude"], row["longitude"])).km,
        axis=1
    )
    return df[df["distance_km"] <= radius_km].sort_values("distance_km")

def get_area_from_message(message):
    message = message.lower()
    for area in area_coordinates:
        if area in message:
            return area
    return None


# System prompt — tells Ollama how to behave
system_prompt = """
You are an offline travel agent for New York City.
You help users find restaurants.

When a user asks for help, give them a more general answer nearby in a 1 km radius of their location. Then ask clarifying questions, such as:
1. What area of NYC are they in? (Midtown, Brooklyn, Queens, etc.)
2. What type of food are they looking for?
3. Food price range?
And give them a more focused answer after each question.

If you can't give an answer, tell that that you dont have that information.

Once you have enough information, give specific recommendations.
Keep responses short and conversational.
"""

# Conversation history — this is what makes it remember context
messages = [{"role": "system", "content": system_prompt}]

print("NYC Travel Agent ready! Type 'quit' to exit.\n")

while True:
    user_input = input("You: ")

    if user_input.lower() == "quit":
        break

    # Check if user mentioned an area
    detected_area = get_area_from_message(user_input)

    if detected_area:
        # Filter real restaurants near that area
        coords = area_coordinates[detected_area]
        nearby = filter_nearby(restaurants, coords, radius_km=1.0)
        restaurant_list = nearby[["name", "cuisine", "distance_km"]].head(20).to_dict(orient="records")

        # Inject real data into the user message
        user_input += f"\n\n[Nearby restaurants from local data: {restaurant_list}]"
        print(f"(Found {len(nearby)} restaurants near {detected_area})")

    messages.append({"role": "user", "content": user_input})

    response = ollama.chat(
        model="llama3.2",
        messages=messages
    )

    assistant_message = response["message"]["content"]
    messages.append({"role": "assistant", "content": assistant_message})

    print(f"\nAgent: {assistant_message}\n")

NYC Travel Agent ready! Type 'quit' to exit.


Agent: Time Square! There are plenty of great options around there.

Can you tell me what type of food you're in the mood for? Are you looking for something classic American, Italian, Asian, or maybe something else?

(I'm happy to give you a general idea of what's nearby, but I'd love to get a bit more specific from you!)


Agent: Time Square is a bit of a melting pot, so I can give you a few ideas.

Can you tell me what's your budget like for dinner? Are you looking to spend under $20, $20-$50, or are you feeling fancy and want to splurge?

(Also, I can give you a list of generic options, but I'd rather try to find something that fits your vibe!)


Agent: Italian food is always a great choice in NYC.

Okay, so we're talking Italian food under $30, and you're near Time Square... Let me see what I can come up with.

There's a great little spot called Carmine's that's known for its pasta dishes and family-style service. It's a bit on the pri

Ok so now I am going to try and break down all the different cell in the notebook to organize it better

Imports

In [42]:
import geopandas as gpd
import ollama
from geopy.distance import geodesic

Load Data

In [43]:
def load_restaurants(path="data/nyc_restaurants.geojson"):
    restaurants = gpd.read_file(path)
    restaurants["latitude"] = restaurants.geometry.to_crs(epsg=3857).centroid.to_crs(epsg=4326).y
    restaurants["longitude"] = restaurants.geometry.to_crs(epsg=3857).centroid.to_crs(epsg=4326).x
    restaurants = restaurants[["name", "cuisine", "addr:street", "latitude", "longitude"]].dropna(subset=["name"])
    return restaurants

restaurants = load_restaurants()
print(f"Loaded {len(restaurants)} restaurants")

Loaded 7635 restaurants


Area Coordinates & Filtering

In [44]:
area_coordinates = {
    "midtown": (40.7549, -73.9840),
    "times square": (40.7580, -73.9855),
    "brooklyn": (40.6782, -73.9442),
    "queens": (40.7282, -73.7949),
    "manhattan": (40.7831, -73.9712),
    "bronx": (40.8448, -73.8648),
}

def filter_nearby(df, user_location, radius_km=1.0):
    df = df.copy()
    df["distance_km"] = df.apply(
        lambda row: geodesic(user_location, (row["latitude"], row["longitude"])).km,
        axis=1
    )
    return df[df["distance_km"] <= radius_km].sort_values("distance_km")

def get_area_from_message(message):
    message = message.lower()
    for area in area_coordinates:
        if area in message:
            return area
    return None

Ollama

In [45]:
system_prompt = """
You are an offline travel agent for New York City.
You help users find restaurants.

When a user asks for help, give them a more general answer nearby in a 1 km radius of their location. Then ask clarifying questions, such as:
1. What area of NYC are they in? (Midtown, Brooklyn, Queens, etc.)
2. What type of food are they looking for?
3. Food price range?
And give them a more focused answer after each question.

If you can't give an answer, tell that that you dont have that information.

Once you have enough information, give specific recommendations.
Keep responses short and conversational.
"""

def ask_ollama(messages):
    response = ollama.chat(
        model="llama3.2",
        messages=messages
    )
    return response["message"]["content"]

Chat Loop (run this when you want to start chatting)

In [46]:
def run_travel_agent():
    messages = [{"role": "system", "content": system_prompt}]

    print("NYC Travel Agent ready! Type 'quit' to exit.\n")

    while True:
        user_input = input("You: ")

        if user_input.lower() == "quit":
            break

        detected_area = get_area_from_message(user_input)

        if detected_area:
            coords = area_coordinates[detected_area]
            nearby = filter_nearby(restaurants, coords, radius_km=1.0)
            restaurant_list = nearby[["name", "cuisine", "distance_km"]].head(20).to_dict(orient="records")
            user_input += f"\n\n[Nearby restaurants from local data: {restaurant_list}]"
            print(f"(Found {len(nearby)} restaurants near {detected_area})")

        messages.append({"role": "user", "content": user_input})
        assistant_message = ask_ollama(messages)
        messages.append({"role": "assistant", "content": assistant_message})

        print(f"\nAgent: {assistant_message}\n")

In [47]:
run_travel_agent()

NYC Travel Agent ready! Type 'quit' to exit.


Agent: I'd love to help you find a great spot to eat in NYC.

Can you tell me where you are right now? Are you in Midtown, Brooklyn, Queens, or somewhere else?

(Found 482 restaurants near times square)

Agent: Let me give you some suggestions nearby.

Here are some restaurants near you in Times Square:

1. Revel & Rye (0.06 km) - known for its craft cocktails and creative American cuisine
2. Junior's (0.10 km) - a classic New York diner with a nostalgic feel
3. Le Marais (0.11 km) - a high-end steakhouse with a sophisticated atmosphere

What type of food are you in the mood for? Are you looking for something specific, like Italian or Mexican?


Agent: I have a few Chinese options for you in Times Square.

Here's one:

1. Silky Kitchen (0.13 km) - a Chinese restaurant with a wide variety of dishes, including Szechuan, Cantonese, and more.

Can you tell me, what's your budget for food? Are you looking for something affordable or willing to 